# Cliff walking: evaluation

Evaluate Q-learning and SARSA checkpoints without training or changing their saved Q-tables. Missing or incompatible models are reported explicitly. See [the training notebook](train_grid_cliff.ipynb) to create new checkpoints.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rl_project").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import Image, Markdown, Video, display
from rl_project.runtime import video_sort_key
from rl_project.discrete import validate_algorithms
ALGORITHM_LABELS = {"qlearning": "Q-learning", "sarsa": "SARSA"}
from pprint import pprint
from rl_project.discrete import latest_checkpoint, load_for_evaluation
from rl_project.runtime import evaluate_policy

## Select checkpoints and evaluation settings

Set an explicit checkpoint for either algorithm, or leave it as `None` to select that algorithm’s latest completed new-format training run. A missing explicit path is reported as unavailable; it does not silently fall back to another model. Relative paths are resolved from the repository root.

The environment is restored from checkpoint metadata and printed before evaluation. `ENV_OVERRIDES` can change the external cutoff; changes to the learned task configuration are rejected. No notebook-wide shared JSON configuration is required.

In [2]:
ENVIRONMENT = "cliff"
ALGORITHMS = ("qlearning", "sarsa")
OUTPUT_ROOT = ROOT / "artifacts"
CHECKPOINT_PATHS = {
    "qlearning": None,  # Optional explicit path to checkpoints/model.npz.
    "sarsa": None,
}
USE_LATEST_COMPLETED = True  # Used only when the corresponding explicit path is None.
EVALUATION_SEED = 1234
EVALUATION_EPISODES = 1
RECORD = True
ENV_OVERRIDES = {}  # By default, restore the task and cutoff saved with each checkpoint.
# Optional external cutoff override: ENV_OVERRIDES = {"max_steps": 200}

## Greedy evaluation

The policy chooses a maximum-Q action without epsilon exploration. The default is one episode per algorithm: repeated deterministic episodes on the same fixed task are not independent evidence. Even `layout="random"` creates one fixed cliff map from its saved `map_seed`; changing the evaluation seed does not generate a new map. Success means completing the task, even if return is negative.

Each evaluation gets its own artifact directory containing the selected checkpoint path, restored environment configuration and per-episode results. If different checkpoints were trained on different tasks, inspect their printed configurations before comparing the outcomes.

In [3]:
results = {}
statuses = {}
selected_checkpoints = {}
for algorithm in validate_algorithms(ALGORITHMS):
    checkpoint = CHECKPOINT_PATHS[algorithm]
    if checkpoint is None and USE_LATEST_COMPLETED:
        checkpoint = latest_checkpoint(ENVIRONMENT, algorithm, OUTPUT_ROOT)
    selected_checkpoints[algorithm] = checkpoint
    agent, status = load_for_evaluation(
        ENVIRONMENT, algorithm, checkpoint, seed=EVALUATION_SEED, env_overrides=ENV_OVERRIDES,
    )
    statuses[algorithm] = status
    display(Markdown(f"### {ALGORITHM_LABELS[algorithm]}: {status}"))
    if agent is None:
        continue
    try:
        print(f"Checkpoint: {agent.source_checkpoint['path']}")
        print("Restored environment settings:")
        pprint(agent.env.config, sort_dicts=False)
        result = evaluate_policy(
            agent.env, agent.greedy_action, ENVIRONMENT, algorithm,
            episodes=EVALUATION_EPISODES, seed=EVALUATION_SEED,
            output_root=OUTPUT_ROOT, record=RECORD,
            config={"checkpoint": agent.source_checkpoint},
        )
        results[algorithm] = result
        headers = ["Episode", "Return", "Steps", "Goal reached", "End reason"]
        if ENVIRONMENT == "flag":
            headers.append("Flags collected")
        table = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
        for row in result.episodes:
            values = [str(row["episode"]), f"{row['reward']:.2f}", str(row["length"]),
                      "Yes" if row["success"] else "No", row["termination_reason"]]
            if ENVIRONMENT == "flag":
                values.append(str(row["flags_collected"]))
            table.append("| " + " | ".join(values) + " |")
        display(Markdown("\n".join(table)))
        print(f"Saved evaluation: {result.run_dir}")
    finally:
        agent.env.close()
if len(results) != len(ALGORITHMS):
    print("Some selected models are unavailable or incompatible; results cover only the available models.")

### Q-learning: available

Checkpoint: /Users/anthonymc/Desktop/NAML_project/artifacts/cliff/qlearning/20260913T101607629849Z_seed0_64ef511c/checkpoints/model.npz
Restored environment settings:
{'render_mode': None,
 'map_seed': 34,
 'n_cliffs': 12,
 'max_steps': 200,
 'layout': 'canonical'}


| Episode | Return | Steps | Goal reached | End reason |
| --- | --- | --- | --- | --- |
| 1 | -17.00 | 17 | Yes | success |

Saved evaluation: /Users/anthonymc/Desktop/NAML_project/artifacts/cliff/qlearning/20260913T101752423789Z_seed1234_76a3ae18


### SARSA: available

Checkpoint: /Users/anthonymc/Desktop/NAML_project/artifacts/cliff/sarsa/20260913T101618017619Z_seed0_e0ba56b5/checkpoints/model.npz
Restored environment settings:
{'render_mode': None,
 'map_seed': 34,
 'n_cliffs': 12,
 'max_steps': 200,
 'layout': 'canonical'}


| Episode | Return | Steps | Goal reached | End reason |
| --- | --- | --- | --- | --- |
| 1 | -23.00 | 23 | Yes | success |

Saved evaluation: /Users/anthonymc/Desktop/NAML_project/artifacts/cliff/sarsa/20260913T101753400212Z_seed1234_3a55189a


## Presentation recordings

With `RECORD=True`, the first evaluation episode for each available algorithm is saved as `episode_0.mp4` in its own `videos/evaluation/` folder. Recordings are capped at 300 frames and 800 pixels wide; evaluation continues after the frame cap. The table above uses the full episode.

In [4]:
for algorithm, result in results.items():
    for path in sorted((result.run_dir / "videos/evaluation").glob("*.mp4"), key=video_sort_key):
        display(Markdown(f"### {ALGORITHM_LABELS[algorithm]} — {path.name}"))
        display(Video(filename=str(path), embed=True))

### Q-learning — episode_0.mp4

### SARSA — episode_0.mp4